# Results for model: microsoft/phi-3-small-128k-instruct

In [ ]:
 import polars as pl
  import xgboost as xgb
  from sklearn.model_selection import train_test_split
  from sklearn.metrics import mean_squared_error

  # Load data
  data = pl.read_parquet('./jane_street_data/train.parquet')

  # Feature Engineering
  global_top_quantile = data.groupby('date_id')['feature_00'].quantile(0.85).to_series()
  data = data.join(global_top_quantile.rename('global_top_quantile'), on='date_id')

  # Create target and features
  X = data.drop('responder_6', axis=1)
  y = data['responder_6']

  # Split data into train and test sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  # Train XGBoost Regressor
  dtrain = xgb.DMatrix(X_train, label=y_train)
  dtest = xgb.DMatrix(X_test, label=y_test)
  params = {
      'objective': 'reg:squarederror',
      'eval_metric': 'rmse',
      'max_depth': 6,
      'eta': 0.1,
      'subsample': 0.8,
      'colsample_bytree': 0.8,
      'seed': 42
  }
  num_round = 100
  bst = xgb.train(params, dtrain, num_round, evals=[(dtest, 'test')])

  # Evaluate model
  y_pred = bst.predict(dtest)
  rmse = mean_squared_error(y_test, y_pred, squared=False)
  print(f'RMSE: {rmse}')